# 01 — Exploratory Data Analysis
## MIT Sloan SSAC 2026 Hackathon

**Mission:** Load the dataset, understand its shape, find the interesting bits, and not panic.

This notebook is your first date with the data. Be curious. Be suspicious. 
Trust nothing. Verify everything. Like a detective, but with histograms.

---

## 1. Setup + Load Data

⚠️ **HACKATHON DAY:** Change the filename below to the actual CSV you received.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

# So we can import our custom scripts — because absolute imports are for people
# who have time to configure proper packaging (we do not)
sys.path.append(os.path.abspath('../scripts'))
from preprocessing import *

sns.set_theme(style='whitegrid')
pio.templates.default = 'plotly_dark'
%matplotlib inline

print('🚀 All imports loaded. Time to science the sh*t out of this data.')

In [ ]:
# ============================================================
# ⬆️⬆️⬆️  CHANGE THE FILENAME BELOW  ⬆️⬆️⬆️
# ============================================================
# Drop your CSV into the data/ folder, then update this path.
# This is literally the only thing you need to change.
# Everything else auto-adapts. Like a cockroach. But useful.

df = load_and_inspect('../data/CHANGEME.csv')

# ============================================================
# ⬆️⬆️⬆️  SERIOUSLY, CHANGE THE FILENAME  ⬆️⬆️⬆️
# ============================================================

## 2. Auto Column Type Detection

Let the algorithm classify your columns. Review the output — it's usually
right, but sometimes it thinks a ZIP code is "numeric_continuous" and
honestly, who among us hasn't made that mistake.

In [ ]:
col_types = identify_column_types(df)

# Print a nice summary
for type_name, cols in col_types.items():
    if cols:
        print(f'\n{type_name} ({len(cols)} columns):')
        for c in cols:
            print(f'  - {c}: {df[c].nunique()} unique, {df[c].isnull().sum()} missing')

## 3. Auto EDA Report

This generates a full interactive HTML report. It takes a minute or two.
Go grab coffee. You've earned it (you've been working for like 5 minutes).

In [ ]:
try:
    from ydata_profiling import ProfileReport
    print('⏳ Generating auto EDA report (this takes a bit, go hydrate)...')
    profile = ProfileReport(df, title='SSAC Fan Data', explorative=True, minimal=True)
    profile.to_file('../outputs/eda_report.html')
    print('✅ Report saved to outputs/eda_report.html — open it in your browser!')
except ImportError:
    print('⚠️ ydata-profiling not installed. Skipping auto report.')
    print('   Run: pip install ydata-profiling')
except Exception as e:
    print(f'⚠️ Auto EDA failed: {e}')
    print('   No worries, we\'ll do it manually below. The old-fashioned way. Like animals.')

## 4. Numeric Distributions

Histograms for every numeric column. Look for:
- Skewed distributions (might need transformation)
- Bimodal distributions (hidden segments!)
- Outliers that look like data entry errors
- Likert scales pretending to be continuous

In [ ]:
print('📊 Numeric Summary:')
numeric_cols = col_types['numeric_continuous'] + col_types['numeric_ordinal']
if numeric_cols:
    display(df[numeric_cols].describe().round(2))
else:
    print('No numeric columns found. This is... unusual.')

In [ ]:
# Grid of histograms — 4 columns wide, auto rows
if numeric_cols:
    n_cols_plot = 4
    n_rows_plot = (len(numeric_cols) + n_cols_plot - 1) // n_cols_plot
    fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(20, 4 * n_rows_plot))
    axes = axes.flatten() if n_rows_plot > 1 else [axes] if n_rows_plot == 1 and n_cols_plot == 1 else axes.flatten()
    
    for idx, col in enumerate(numeric_cols):
        ax = axes[idx]
        df[col].hist(bins=30, ax=ax, color='#4ECDC4', edgecolor='white', alpha=0.8)
        ax.set_title(col, fontsize=10)
        ax.set_ylabel('Count')
    
    # Hide empty subplots
    for idx in range(len(numeric_cols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Numeric Distributions', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()
    print('👆 Look for bimodal distributions — those might be your natural segments!')

## 5. Categorical Breakdowns

Value counts and bar charts for categorical columns. Look for:
- Dominant categories (>80% one value = not very useful for clustering)
- Rare categories (might want to bucket these)
- Categories that suggest natural segments ("hardcore" vs "casual" etc.)

In [ ]:
cat_cols = col_types['categorical_low'] + col_types['categorical_high'] + col_types['binary']

for col in cat_cols:
    if col in df.columns:
        print(f'\n--- {col} ({df[col].nunique()} unique) ---')
        vc = df[col].value_counts()
        print(vc.head(10))
        
        # Bar chart for top categories
        if df[col].nunique() <= 20:
            fig = px.bar(
                x=vc.head(10).index.astype(str),
                y=vc.head(10).values,
                title=f'{col} — Top Categories',
                labels={'x': col, 'y': 'Count'},
                color_discrete_sequence=['#FF6B6B'],
            )
            fig.show()

## 6. Missing Data Visualization

If the data's perfect, skip this. (Spoiler: the data is never perfect.)

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('pct', ascending=False)

if len(missing_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(missing_df) * 0.3)))
    
    # Bar chart of missing %
    axes[0].barh(missing_df.index, missing_df['pct'], color='#FF6B6B')
    axes[0].set_xlabel('% Missing')
    axes[0].set_title('Missing Values by Column')
    
    # Heatmap of missing pattern
    sns.heatmap(df[missing_df.index].isnull().T, cbar=False, yticklabels=True, ax=axes[1],
                cmap=['#1a1a2e', '#FF6B6B'])
    axes[1].set_title('Missing Data Pattern (red = missing)')
    
    plt.tight_layout()
    plt.show()
else:
    print('🎉 Zero missing values! Either the data is clean or something is terribly wrong.')

## 7. Correlation Analysis

Correlation heatmap + top correlated pairs. Look for:
- Highly correlated features (r > 0.8) — might be redundant
- Unexpected correlations — "wait, spending correlates with WHAT?"
- Zero/near-zero correlations with everything — low-info features

In [ ]:
if len(numeric_cols) > 1:
    corr = df[numeric_cols].corr()
    
    # Heatmap
    fig, ax = plt.subplots(figsize=(min(20, len(numeric_cols)), min(16, len(numeric_cols) * 0.8)))
    mask = np.triu(np.ones_like(corr, dtype=bool))  # Hide upper triangle (redundant)
    sns.heatmap(corr, mask=mask, annot=len(numeric_cols) <= 15, fmt='.2f',
                cmap='RdBu_r', center=0, ax=ax, linewidths=0.5)
    ax.set_title('Correlation Matrix', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Top 10 strongest correlations (excluding self-correlation)
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    pairs = upper.unstack().dropna().abs().sort_values(ascending=False)
    print('\n🔝 Top 10 Strongest Correlations:')
    for (col1, col2), val in pairs.head(10).items():
        actual = corr.loc[col1, col2]
        direction = '↗️' if actual > 0 else '↘️'
        print(f'   {direction} {col1} × {col2}: r = {actual:.3f}')
else:
    print('Not enough numeric columns for correlation analysis.')

## 8. Key Variable Deep Dives

These are placeholder cells. On hackathon day, uncomment and adapt
based on what columns you actually have. Think of these as pre-written
analysis templates — just add column names and stir.

In [ ]:
# ======================================
# SPENDING VARIABLES — uncomment and adapt
# ======================================
# Replace with actual column names from your dataset
#
# spend_cols = ['ticket_spend', 'merch_spend', 'food_spend']  # CHANGE THESE
# 
# print('💰 Spending Summary:')
# display(df[spend_cols].describe())
# 
# # Total spend distribution
# df['total_spend'] = df[spend_cols].sum(axis=1)
# fig = px.histogram(df, x='total_spend', nbins=50, title='Total Spending Distribution',
#                    color_discrete_sequence=['#4ECDC4'])
# fig.show()
# 
# # Spend by category (if applicable)
# # demo_col = 'gender'  # CHANGE THIS
# # fig = px.box(df, x=demo_col, y='total_spend', title=f'Spending by {demo_col}')
# # fig.show()

In [ ]:
# ======================================
# ENGAGEMENT VARIABLES — uncomment and adapt
# ======================================
# Replace with actual column names from your dataset
#
# engage_cols = ['games_attended', 'social_follows', 'app_sessions']  # CHANGE THESE
# 
# print('📱 Engagement Summary:')
# display(df[engage_cols].describe())
# 
# # Cross-tab engagement vs spending
# # fig = px.scatter(df, x='games_attended', y='total_spend',
# #                  title='Attendance vs Spending', trendline='ols')
# # fig.show()

In [ ]:
# ======================================
# DEMOGRAPHIC SPLITS — uncomment and adapt
# ======================================
# Replace with actual column names from your dataset
#
# demo_col = 'age_group'  # CHANGE THIS
# metric_col = 'total_spend'  # CHANGE THIS
# 
# # Grouped bar chart
# grouped = df.groupby(demo_col)[metric_col].mean().sort_values(ascending=False)
# fig = px.bar(x=grouped.index, y=grouped.values,
#              title=f'Mean {metric_col} by {demo_col}',
#              color_discrete_sequence=['#45B7D1'])
# fig.show()
# 
# # Cross-tab with another categorical
# # second_demo = 'fan_type'  # CHANGE THIS
# # ct = pd.crosstab(df[demo_col], df[second_demo], normalize='index') * 100
# # display(ct.round(1))

## 9. Initial Observations

### Key observations from EDA:
1. 
2. 
3. 

### Hypotheses for clustering:
1. 
2. 
3. 

### Features to prioritize for clustering:
- 
- 
- 

### Red flags / data quality issues:
- 
- 

---
*Fill this in during EDA. Future-you will thank present-you for taking notes.*
*Past-you would have left this blank. Don't be past-you.*